# C1 — Can binding be learned *across* domains? (Stage 3)

**Question.** Train a binding head on two ddPCA domains and predict the third. Does binding
transfer?

**What it does.** Leave-one-domain-out over GRB2-SH3, PSD95-PDZ3 and GB1. The binding head sees a
64-component PCA of the frozen features (a small head, because ~2,800 measurements against a
650M-parameter encoder would memorise).

**Why it fails, and why that was predictable.** To predict GRB2-SH3 the model trains on a PDZ
domain that reads C-terminal peptides and on GB1, which binds a folded IgG surface. No training
domain shares the SH3 binding mode, so there is nothing to transfer. This experiment is kept
because it is the first step of the reasoning, not because it tests SH3 biology.

**Two versions.**
- `original` — the Stage 3 code exactly, which (it turned out) **stopped training on the test
  domain**: the held-out domain doubled as the early-stopping set. That is leakage.
- `corrected` — early stopping on an inner split of the *training* domains only, with the
  variance-floored scaler and whitened PCA established in Stage 3b.

**Inputs:** A2 features. **Runtime:** about 3 min, T4 GPU. **Outputs:** `lodo_results.csv`, `RESULTS_C1.md`.
Explanation: `docs/C1_binding_cross_domain.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'C1', 'Binding across domains (leave-one-domain-out)', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

In [ ]:
DD = pd.read_parquet(R.res('A2', 'features_ddpca.parquet'))
EMB = C.load_embeddings(R.res('A2', 'esm_domain_embeddings.npz'))
Xd, DB = C.build_X(DD, EMB)
B = DB[DB.ddG_bind.notna() & DB.binding_reliable.eq(True)].copy()
XB = Xd[B.index.values].astype(np.float32); B = B.reset_index(drop=True)
print(B.groupby('domain_key').agg(measurements=('pos', 'size'), positions=('pos', 'nunique')).to_string())

In [ ]:
from sklearn.decomposition import PCA
y_all = B.ddG_bind.values.astype(np.float32); rows = []
for held in B.domain_key.unique():
    te = (B.domain_key == held).values; tr = ~te; y = y_all[te]; esm = C.spearman(y, B.esm_score.values[te])
    # original Stage 3: raw PCA, 1e-6 scaler, the TEST domain used for early stopping
    pca = PCA(n_components=64, random_state=0).fit(XB[tr])
    s2 = C.ScalerStage3().fit(pca.transform(XB[tr]))
    Ztr, Zte = s2(pca.transform(XB[tr])), s2(pca.transform(XB[te]))
    p = np.mean([C.predict(C.train(C.BindHead, Ztr, y_all[tr], Zte, y, epochs=60, bs=256, lr=3e-3, wd=1e-3, seed=s), Zte)
                 for s in range(5)], 0)
    lo, hi = C.boot_ci(y, p, 1000)
    rows.append(dict(mode='original', held_out=held, n=int(te.sum()), rho=C.spearman(y, p), lo=lo, hi=hi, esm=esm))
    # corrected: early stopping on an inner split of the training domains, Stage 3b preprocessing
    gtr = (B.domain_key[tr] + '_' + B.pos[tr].astype(str)).values
    vm = C.inner_validation(gtr, 0); itr = np.where(tr)[0]; fit, val = itr[~vm], itr[vm]
    s0 = C.Scaler().fit(XB[fit]); pca = PCA(n_components=64, whiten=True, random_state=0).fit(s0(XB[fit]))
    proj = lambda Q: np.clip(pca.transform(s0(Q)), -10, 10).astype(np.float32)
    ms = [C.train(C.BindHead, proj(XB[fit]), y_all[fit], proj(XB[val]), y_all[val], epochs=60, bs=256, seed=s, guard=True)
          for s in range(5)]
    p = np.mean([C.predict(m, proj(XB[te])) for m in ms if m is not None], 0)
    lo, hi = C.boot_ci(y, p, 1000)
    rows.append(dict(mode='corrected', held_out=held, n=int(te.sum()), rho=C.spearman(y, p), lo=lo, hi=hi, esm=esm))
LD = pd.DataFrame(rows); LD['beats_esm'] = LD.rho.abs() > LD.esm.abs(); LD.to_csv(R.out('lodo_results.csv'), index=False)
print(LD.round(3).to_string(index=False))

In [ ]:
for _, r in LD.iterrows():
    for c in ('rho', 'esm', 'n', 'lo', 'hi'): R.metric(f'{r["mode"]}.{r.held_out}.{c}', r[c])
g = LD[LD.held_out == 'GRB2-SH3'].set_index('mode')
R.section('Leave-one-domain-out', 'Spearman ρ on the held-out domain, 95% bootstrap CI. ESM = zero-shot score.', LD, '{:.3f}')
R.finish([f"Held-out GRB2-SH3: head ρ {g.loc['original','rho']:+.3f} (original) / {g.loc['corrected','rho']:+.3f} (corrected) "
          f"against zero-shot ESM {g.loc['original','esm']:+.3f}.",
          f"The head beats ESM on {int(LD[LD['mode']=='corrected'].beats_esm.sum())}/3 held-out domains in the corrected version.",
          'Transfer between unrelated binding modes fails; the next question (C2) removes transfer altogether.'])